# Notebook 9: Super-Resolution Test (2km -> 250m, guided by MODIS VCF C6 2020)

**Goal:** downscale the coarse (2km) PACE-VCF predictions to 250m using the
scientifically-validated MODIS VCF Collection 6 (2020) product as a guide,
producing a MODIS-VCF-*like* fine-resolution product for years where a real
MODIS VCF doesn't exist (the eventual target: 2026 PACE data with only
"previous" -- 2020 -- MODIS VCF available as guide, since a same-year
reference will never exist operationally).

## Why this isn't just "run a guided filter and check the output"

Reference-guided super-resolution injects the guide's own fine spatial
texture into the output. If you evaluate the result against the *same* guide
data you fed in, the comparison is circular by construction -- it'll look
good regardless of whether the method generalizes, because you literally put
the answer in. Two holdout layers guard against that here, matching the
gap that will exist operationally (no same-year MODIS VCF at all):

1. **Region-stratified tile split** (guide tiles vs. test tiles). Different
   biomes behave differently (arctic vs. desert vs. tropical canopy
   structure), so both sets need every represented region, not just a random
   split that could leave a region entirely in one bucket. No real
   biome/ecoregion dataset was available on this system, so regions are
   derived from each tile's own geography (centroid latitude band x rough
   continent) -- a coarse proxy, not a true ecoregion classification; swap in
   a real one later if it matters.
   - **Guide tiles**: used to tune the guided filter's hyperparameters
     (window radius, epsilon) against held-out checkerboard blocks (see
     below) -- a development set, not directly reported as the result.
   - **Test tiles**: touched only once, at the end, with the
     already-chosen hyperparameters -- the actual reported result, broken
     out by region.

2. **Within-tile checkerboard block holdout.** A guided filter fundamentally
   needs *some* fine-resolution guide value at every pixel it's asked to
   refine, so "never touch a test tile's 250m data" isn't possible the way a
   whole-tile holdout would be for a purely-learned method. Instead, each
   tile's 2km-cell blocks are marked visible/held-out in a checkerboard
   pattern: held-out blocks get the guide fed only the coarse-upsampled
   value (i.e. no fine detail at all at those locations, same as plain
   upsampling), and the real MODIS 250m value there is compared against the
   guided-filter output *only* after the fact -- so no evaluation location
   ever had its own real fine-resolution value available to the method that
   produced its prediction.

## Method

[Fast Guided Filter](https://arxiv.org/abs/1505.00996) (He & Sun): a local
linear model between the guide (fine) and target (coarse, upsampled to fine
grid) within a moving window. Implemented here in plain numpy/scipy, no
exotic dependencies. This is a first-draft/default choice -- swap in a
different method (e.g. ratio/residual injection) later if guided filter
underperforms; the region-split and checkerboard-holdout evaluation harness
below doesn't depend on which method fills in `apply_super_resolution()`.

**Before running:** no new setup -- reads existing files from the inference
directory (`PACE_VCF_2025_{tile}_2km.tif`, `MODIS_VCF_C6_2020_{tile}_250m.tif`),
which nearly all production tiles already have from prior inference runs.


In [ ]:
# ## Cell 1: CONFIGURATION
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

from pathlib import Path

INFERENCE_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/inference")
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/superres")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PACE_YEAR = 2025
MODIS_REFERENCE_YEAR = 2020  # last scientifically-validated MODIS VCF (C6)

# Full production tile list (matches TILES_TO_PROCESS elsewhere in the
# pipeline) -- actual availability is checked at load time below, since not
# every tile may have finished inference yet.
ALL_TILES = [
    "h08v04", "h08v05", "h09v04", "h09v05", "h10v04", "h10v05", "h10v06",
    "h11v02", "h11v03", "h11v04", "h11v05", "h11v08", "h11v09", "h11v10",
    "h12v01", "h12v02", "h12v03", "h12v04", "h12v05", "h12v09", "h12v10",
    "h12v12", "h13v01", "h13v02", "h13v10", "h13v11", "h13v12", "h16v01",
    "h17v05", "h18v03", "h18v04", "h18v07", "h19v04", "h19v07", "h19v08",
    "h19v09", "h19v10", "h19v11", "h19v12", "h20v02", "h20v03", "h20v04",
    "h20v06", "h20v08", "h20v09", "h20v10", "h20v11", "h21v01", "h21v02",
    "h21v04", "h21v05", "h21v06", "h21v10", "h22v03", "h22v04", "h23v02",
    "h23v03", "h24v02", "h24v03", "h24v04", "h26v06", "h27v04", "h27v06",
    "h27v07", "h28v11", "h29v11", "h29v12", "h30v12", "h31v11",
]

# Grid parameters (matches the rest of the PACE-VCF pipeline)
TILE_SIZE_PACE = 600     # coarse (2km) grid
TILE_SIZE_MODIS = 4800   # fine (250m) grid
AGGREGATION_FACTOR = TILE_SIZE_MODIS // TILE_SIZE_PACE  # 8

NO_DATA_OUT = 255  # matches write_vcf_geotiff's uint8 convention in 7f/7e

MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677
MODIS_TILE_SIZE_M = 1111950.5196666666
MODIS_SPHERE_RADIUS = 6371007.181

# Region-stratified guide/test split
GUIDE_FRACTION = 0.7
RANDOM_STATE = 42

# Checkerboard holdout: block size in COARSE (2km) cells -- a 1x1 checkerboard
# alternates individual 2km cells; larger values alternate bigger blocks.
CHECKERBOARD_BLOCK_SIZE = 1

# Guided filter hyperparameter search grid (tuned on guide tiles only)
GF_RADIUS_CANDIDATES = [1, 2, 3, 4]
GF_EPS_CANDIDATES = [1000.0, 10000.0, 100000.0]

print("Configuration loaded")
print(f"  Inference dir: {INFERENCE_DIR}")
print(f"  PACE year (coarse prediction): {PACE_YEAR}")
print(f"  MODIS reference year (fine guide): {MODIS_REFERENCE_YEAR}")
print(f"  Coarse grid: {TILE_SIZE_PACE}x{TILE_SIZE_PACE}, fine grid: {TILE_SIZE_MODIS}x{TILE_SIZE_MODIS}")
print(f"  Guide fraction: {GUIDE_FRACTION}")


In [ ]:
# ## Cell 2: Imports

import logging
import numpy as np
import rasterio
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print("Imports complete")


In [ ]:
# ## Cell 3: Projection + Region Classification Functions

def get_tile_bounds_sinusoidal(tile: str):
    """Tile bounds in sinusoidal coordinates (min_x, min_y, max_x, max_y)."""
    h = int(tile[1:3])
    v = int(tile[4:6])
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_x = min_x + MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    min_y = max_y - MODIS_TILE_SIZE_M
    return (min_x, min_y, max_x, max_y)


def sinusoidal_to_latlon(x, y):
    lat = np.degrees(y / MODIS_SPHERE_RADIUS)
    lon = np.degrees(x / (MODIS_SPHERE_RADIUS * np.cos(np.radians(lat))))
    return lat, lon


def get_tile_centroid_latlon(tile: str):
    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)
    return sinusoidal_to_latlon((min_x + max_x) / 2, (min_y + max_y) / 2)


def classify_region(lat: float, lon: float) -> str:
    """
    Coarse geographic proxy for biome, built from tile centroid lat/lon --
    NOT a real ecoregion classification (see notebook intro). Latitude band
    x rough continent, e.g. "Arctic-Americas", "Tropical-AsiaPacific".
    """
    abs_lat = abs(lat)
    if abs_lat >= 55:
        lat_band = "Arctic"
    elif abs_lat >= 35:
        lat_band = "Temperate"
    elif abs_lat >= 20:
        lat_band = "Subtropical"
    else:
        lat_band = "Tropical"

    if -170 <= lon < -30:
        lon_band = "Americas"
    elif -30 <= lon < 60:
        lon_band = "EuropeAfrica"
    else:
        lon_band = "AsiaPacific"

    return f"{lat_band}-{lon_band}"

print("Projection + region classification functions defined")


In [ ]:
# ## Cell 4: Build Tile Region Table + Stratified Guide/Test Split

def find_available_tiles(tiles: list) -> list:
    """Only keep tiles that actually have both the coarse prediction and the
    250m MODIS C6 2020 reference on disk."""
    available = []
    for tile in tiles:
        pred_path = INFERENCE_DIR / f"PACE_VCF_{PACE_YEAR}_{tile}_2km.tif"
        ref_path = INFERENCE_DIR / f"MODIS_VCF_C6_{MODIS_REFERENCE_YEAR}_{tile}_250m.tif"
        if pred_path.exists() and ref_path.exists():
            available.append(tile)
    return available


AVAILABLE_TILES = find_available_tiles(ALL_TILES)
logger.info(f"Available tiles (prediction + 2020 C6 reference both present): "
            f"{len(AVAILABLE_TILES)}/{len(ALL_TILES)}")
missing = sorted(set(ALL_TILES) - set(AVAILABLE_TILES))
if missing:
    logger.warning(f"  Missing (inference or reference not yet available): {missing}")

tile_regions = {}
for tile in AVAILABLE_TILES:
    lat, lon = get_tile_centroid_latlon(tile)
    tile_regions[tile] = classify_region(lat, lon)

from collections import defaultdict
region_to_tiles = defaultdict(list)
for tile, region in tile_regions.items():
    region_to_tiles[region].append(tile)

print(f"\nRegion distribution ({len(region_to_tiles)} regions):")
for region, tiles in sorted(region_to_tiles.items()):
    print(f"  {region:<25} {len(tiles):>3} tiles: {tiles}")


def stratified_guide_test_split(region_to_tiles: dict, guide_fraction: float, random_state: int):
    rng = np.random.default_rng(random_state)
    guide_tiles, test_tiles = [], []

    for region, tiles in sorted(region_to_tiles.items()):
        tiles = list(tiles)
        rng.shuffle(tiles)
        if len(tiles) == 1:
            # Can't split a singleton -- put it in test so the region still
            # gets a genuine held-out evaluation, at the cost of no
            # region-specific guide-tile representation.
            test_tiles.append(tiles[0])
            logger.warning(f"  Region '{region}' has only 1 tile -- assigned to TEST "
                           f"(no guide-tile representation for this region)")
            continue
        n_guide = max(1, round(len(tiles) * guide_fraction))
        n_guide = min(n_guide, len(tiles) - 1)  # always leave >=1 for test
        guide_tiles.extend(tiles[:n_guide])
        test_tiles.extend(tiles[n_guide:])

    return sorted(guide_tiles), sorted(test_tiles)


GUIDE_TILES, TEST_TILES = stratified_guide_test_split(region_to_tiles, GUIDE_FRACTION, RANDOM_STATE)

print(f"\nGuide tiles ({len(GUIDE_TILES)}): {GUIDE_TILES}")
print(f"Test tiles  ({len(TEST_TILES)}): {TEST_TILES}")

print("\nPer-region guide/test counts:")
for region, tiles in sorted(region_to_tiles.items()):
    n_guide = sum(1 for t in tiles if t in GUIDE_TILES)
    n_test = sum(1 for t in tiles if t in TEST_TILES)
    print(f"  {region:<25} guide={n_guide:>2}  test={n_test:>2}")


In [ ]:
# ## Cell 5: Data Loading

def load_coarse_prediction(tile: str) -> np.ndarray:
    """PACE_VCF_{year}_{tile}_2km.tif -> (600, 600) float32, NaN for NoData."""
    path = INFERENCE_DIR / f"PACE_VCF_{PACE_YEAR}_{tile}_2km.tif"
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
    data[data == NO_DATA_OUT] = np.nan
    return data


def load_fine_reference(tile: str) -> np.ndarray:
    """MODIS_VCF_C6_{year}_{tile}_250m.tif -> (4800, 4800) float32, NaN for NoData."""
    path = INFERENCE_DIR / f"MODIS_VCF_C6_{MODIS_REFERENCE_YEAR}_{tile}_250m.tif"
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
    data[data == NO_DATA_OUT] = np.nan
    return data


def upsample_nearest(coarse: np.ndarray, factor: int) -> np.ndarray:
    """(600, 600) -> (4800, 4800) via simple pixel repetition."""
    return np.repeat(np.repeat(coarse, factor, axis=0), factor, axis=1)

print("Data loading functions defined")


In [ ]:
# ## Cell 6: Checkerboard Holdout + Guided Filter

def make_checkerboard_mask(tile_size_coarse: int, factor: int, block_size: int) -> np.ndarray:
    """
    Boolean array at FINE resolution, True = visible (usable as guide input),
    False = held out (guide value hidden; real value only used for scoring
    afterward). Alternates at the scale of `block_size` coarse (2km) cells,
    so an entire 2km cell's worth of fine pixels is visible or held out
    together -- holding out scattered individual fine pixels within an
    otherwise-visible 2km cell would make the local guided-filter window
    trivially fill them in from immediate neighbors, understating error.
    """
    coarse_rows = np.arange(tile_size_coarse) // block_size
    coarse_cols = np.arange(tile_size_coarse) // block_size
    coarse_checkerboard = (coarse_rows[:, None] % 2) == (coarse_cols[None, :] % 2)
    return np.repeat(np.repeat(coarse_checkerboard, factor, axis=0), factor, axis=1)


def box_filter(img, radius):
    valid = np.isfinite(img).astype(np.float32)
    filled = np.where(valid.astype(bool), img, 0.0)
    size = 2 * radius + 1
    sum_vals = uniform_filter(filled, size=size, mode="reflect") * size**2
    sum_valid = uniform_filter(valid, size=size, mode="reflect") * size**2
    with np.errstate(invalid="ignore", divide="ignore"):
        result = sum_vals / sum_valid
    result[sum_valid == 0] = np.nan
    return result



def guided_filter(guide: np.ndarray, target: np.ndarray, radius: int, eps: float) -> np.ndarray:
    """
    Fast Guided Filter (He & Sun): local linear model target ~= a*guide + b
    within a (2*radius+1) window, solved by least squares per window via box
    filters. guide/target must be the same shape (fine resolution), with no
    NaNs (fill before calling).
    """
    mean_g = box_filter(guide, radius)
    mean_t = box_filter(target, radius)
    mean_gt = box_filter(guide * target, radius)
    cov_gt = mean_gt - mean_g * mean_t
    mean_gg = box_filter(guide * guide, radius)
    var_g = mean_gg - mean_g * mean_g

    a = cov_gt / (var_g + eps)
    b = mean_t - a * mean_g

    mean_a = box_filter(a, radius)
    mean_b = box_filter(b, radius)

    return mean_a * guide + mean_b


def apply_super_resolution(coarse_pred: np.ndarray, fine_reference: np.ndarray,
                            visible_mask: np.ndarray, radius: int, eps: float) -> np.ndarray:
    """
    Guided-filter super-resolution with checkerboard holdout. `fine_reference`
    is only used at `visible_mask` locations; held-out locations fall back to
    the coarse-upsampled value (no fine detail injected there), so the guide
    input never contains the answer at any location being scored.
    """
    factor = coarse_pred.shape[0] // TILE_SIZE_PACE * AGGREGATION_FACTOR         if coarse_pred.shape[0] != TILE_SIZE_MODIS else 1  # (coarse_pred is always 600x600 here)
    coarse_upsampled = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    guide = np.where(visible_mask, fine_reference, coarse_upsampled)
    # Guided filter can't handle NaN -- fill remaining gaps (e.g. NoData in
    # the reference even at visible locations) with the coarse-upsampled
    # value as a neutral fallback.
    guide = np.where(np.isfinite(guide), guide, coarse_upsampled)
    target = np.where(np.isfinite(coarse_upsampled), coarse_upsampled, np.nanmean(coarse_pred))

    result = guided_filter(guide, target, radius, eps)

    # print("NaN in guide:", np.isnan(guide).mean())
    # print("NaN in target:", np.isnan(target).mean())
    # print("NaN in sr_result:", np.isnan(result).mean())
    
    return np.clip(result, 0, 100)

print("Checkerboard holdout + guided filter functions defined")


In [ ]:
# ## Cell 7: Evaluation Metrics

def evaluate_holdout(tile: str, radius: int, eps: float, block_size: int = CHECKERBOARD_BLOCK_SIZE):
    """
    Run the checkerboard-holdout super-resolution + evaluation for one tile.
    Returns dict of metrics computed ONLY at held-out (never-visible-to-the-
    method) locations, or None if there's no usable data.
    """
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)

    visible_mask = make_checkerboard_mask(TILE_SIZE_PACE, AGGREGATION_FACTOR, block_size)
    held_out_mask = ~visible_mask

    sr_result = apply_super_resolution(coarse_pred, fine_reference, visible_mask, radius, eps)

    valid = held_out_mask & np.isfinite(fine_reference) & np.isfinite(sr_result)
    n_valid = int(np.sum(valid))
    if n_valid == 0:
        return None

    truth = fine_reference[valid]
    pred = sr_result[valid]
    residual = pred - truth

    # Baseline for comparison: plain nearest-neighbor upsample with no guide
    # detail at all, scored at the same held-out locations
    baseline = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)[valid]
    baseline_residual = baseline - truth

    return {
        "tile": tile,
        "n_valid": n_valid,
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "mae": float(np.mean(np.abs(residual))),
        "bias": float(np.mean(residual)),
        "baseline_rmse": float(np.sqrt(np.mean(baseline_residual ** 2))),
        "baseline_mae": float(np.mean(np.abs(baseline_residual))),
    }

print("Evaluation function defined")


In [ ]:
# ## Cell 8: Tune Guided Filter Hyperparameters on Guide Tiles

print("=" * 70)
print("HYPERPARAMETER SEARCH (guide tiles only)")
print("=" * 70)

search_results = []
for radius in GF_RADIUS_CANDIDATES:
    for eps in GF_EPS_CANDIDATES:
        tile_metrics = []
        for tile in GUIDE_TILES:
            m = evaluate_holdout(tile, radius, eps)
            if m is not None:
                tile_metrics.append(m)
        if not tile_metrics:
            continue
        mean_rmse = float(np.mean([m["rmse"] for m in tile_metrics]))
        mean_baseline_rmse = float(np.mean([m["baseline_rmse"] for m in tile_metrics]))
        search_results.append({
            "radius": radius, "eps": eps,
            "mean_rmse": mean_rmse, "mean_baseline_rmse": mean_baseline_rmse,
            "n_tiles": len(tile_metrics),
        })
        print(f"  radius={radius:>3} eps={eps:>6.1f}  "
              f"RMSE={mean_rmse:6.2f}%  (baseline={mean_baseline_rmse:6.2f}%)  "
              f"n_tiles={len(tile_metrics)}")

best = min(search_results, key=lambda r: r["mean_rmse"])
BEST_RADIUS, BEST_EPS = best["radius"], best["eps"]

best_rmse = best["mean_rmse"]
best_baseline_rmse = best["mean_baseline_rmse"]
print(f"\nBest on guide tiles: radius={BEST_RADIUS}, eps={BEST_EPS} "
      f"(RMSE={best_rmse:.2f}% vs. baseline {best_baseline_rmse:.2f}%)")


In [ ]:
# ## Cell 9: Final Evaluation on Test Tiles (never touched during tuning)

print("=" * 70)
print(f"FINAL EVALUATION ON TEST TILES (radius={BEST_RADIUS}, eps={BEST_EPS})")
print("=" * 70)

test_results = []
for tile in TEST_TILES:
    m = evaluate_holdout(tile, BEST_RADIUS, BEST_EPS)
    if m is None:
        logger.warning(f"  {tile}: no usable held-out data, skipped")
        continue
    m["region"] = tile_regions[tile]
    test_results.append(m)
    print(f"  {tile:<8} ({m['region']:<22}) RMSE={m['rmse']:6.2f}%  "
          f"(baseline {m['baseline_rmse']:6.2f}%)  MAE={m['mae']:6.2f}%  bias={m['bias']:+6.2f}%")

print(f"\n{'='*70}")
print("OVERALL (test tiles)")
print(f"{'='*70}")
overall_rmse = float(np.mean([m["rmse"] for m in test_results]))
overall_baseline_rmse = float(np.mean([m["baseline_rmse"] for m in test_results]))
print(f"  Mean RMSE:          {overall_rmse:.2f}%")
print(f"  Mean baseline RMSE: {overall_baseline_rmse:.2f}%  (plain upsample, no guide detail)")
print(f"  Improvement:        {overall_baseline_rmse - overall_rmse:+.2f} percentage points")

print(f"\n{'='*70}")
print("BY REGION")
print(f"{'='*70}")
by_region = defaultdict(list)
for m in test_results:
    by_region[m["region"]].append(m)
for region, ms in sorted(by_region.items()):
    r_rmse = np.mean([m["rmse"] for m in ms])
    r_baseline = np.mean([m["baseline_rmse"] for m in ms])
    print(f"  {region:<25} n={len(ms):>2}  RMSE={r_rmse:6.2f}%  (baseline {r_baseline:6.2f}%)")


In [ ]:
# ## Cell 10: Visualize One Example Tile

def find_high_contrast_crop(fine_reference: np.ndarray, crop_coarse_size: int = 8):
    """
    Locate the crop_coarse_size x crop_coarse_size (PACE-pixel) window with
    the highest local variance in the real MODIS reference -- i.e. the most
    visually interesting area (forest/water edges, rivers, clearings), so the
    example plot doesn't land on a flat, low-contrast patch. Returns
    (crop_row, crop_col) in coarse (2km) pixel units, or None if there's no
    valid data at all.
    """
    window = crop_coarse_size * AGGREGATION_FACTOR  # fine pixels

    valid = np.isfinite(fine_reference)
    if not valid.any():
        return None

    filled = np.where(valid, fine_reference, 0.0).astype(np.float32)
    valid_f = valid.astype(np.float32)

    sum_x = uniform_filter(filled, size=window, mode="constant") * window ** 2
    sum_x2 = uniform_filter(filled ** 2, size=window, mode="constant") * window ** 2
    count = uniform_filter(valid_f, size=window, mode="constant") * window ** 2

    with np.errstate(invalid="ignore", divide="ignore"):
        mean = sum_x / count
        local_var = sum_x2 / count - mean ** 2

    # Only consider windows that are almost entirely valid data -- a
    # near-empty window can spuriously look "high variance" from noise.
    local_var = np.where(count >= window ** 2 * 0.9, local_var, -1.0)

    best_row_fine, best_col_fine = np.unravel_index(np.argmax(local_var), local_var.shape)
    crop_row = (best_row_fine - window // 2) // AGGREGATION_FACTOR
    crop_col = (best_col_fine - window // 2) // AGGREGATION_FACTOR
    return int(crop_row), int(crop_col)


def plot_example(tile: str, radius: int = BEST_RADIUS, eps: float = BEST_EPS,
                  block_size: int = CHECKERBOARD_BLOCK_SIZE,
                  crop_coarse_size: int = 8, crop_row: int = None, crop_col: int = None):
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)
    visible_mask = make_checkerboard_mask(TILE_SIZE_PACE, AGGREGATION_FACTOR, block_size)
    sr_result = apply_super_resolution(coarse_pred, fine_reference, visible_mask, radius, eps)
    baseline = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    # Default crop: highest-local-variance window in the real MODIS reference
    # (forest/water edges, rivers, clearings), not just centered on valid data.
    if crop_row is None or crop_col is None:
        found = find_high_contrast_crop(fine_reference, crop_coarse_size)
        if found is not None:
            crop_row, crop_col = found
        else:
            crop_row = (TILE_SIZE_PACE - crop_coarse_size) // 2
            crop_col = (TILE_SIZE_PACE - crop_coarse_size) // 2
    crop_row = max(0, min(crop_row, TILE_SIZE_PACE - crop_coarse_size))
    crop_col = max(0, min(crop_col, TILE_SIZE_PACE - crop_coarse_size))

    fine_row0 = crop_row * AGGREGATION_FACTOR
    fine_row1 = (crop_row + crop_coarse_size) * AGGREGATION_FACTOR
    fine_col0 = crop_col * AGGREGATION_FACTOR
    fine_col1 = (crop_col + crop_coarse_size) * AGGREGATION_FACTOR

    coarse_crop = coarse_pred[crop_row:crop_row + crop_coarse_size, crop_col:crop_col + crop_coarse_size]
    baseline_crop = baseline[fine_row0:fine_row1, fine_col0:fine_col1]
    sr_crop = sr_result[fine_row0:fine_row1, fine_col0:fine_col1]
    ref_crop = fine_reference[fine_row0:fine_row1, fine_col0:fine_col1]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    im0 = axes[0].imshow(coarse_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[0].set_title(f"{tile}: Coarse prediction (2km)\n{crop_coarse_size}x{crop_coarse_size} PACE pixels")
    im1 = axes[1].imshow(baseline_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[1].set_title("Baseline (nearest upsample)")
    im2 = axes[2].imshow(sr_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[2].set_title(f"Super-resolved (r={radius}, eps={eps})")
    im3 = axes[3].imshow(ref_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[3].set_title(f"Real MODIS C6 {MODIS_REFERENCE_YEAR} (250m)")
    for ax, im in zip(axes, [im0, im1, im2, im3]):
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="% Tree Cover")
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f"superres_example_{tile}_crop_r{crop_row}_c{crop_col}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    logger.info(f"Saved: {fig_path}")
    plt.show()


if TEST_TILES:
    plot_example(TEST_TILES[0])

In [ ]:
# ## Cell 11: Residual/Ratio Injection ("pan-sharpening style") Method

def apply_super_resolution_residual(coarse_pred: np.ndarray, fine_reference: np.ndarray,
                                     visible_mask: np.ndarray, radius: int) -> np.ndarray:
    """
    Ratio injection (SFIM-style pan-sharpening): multiply the coarse-upsampled
    target by the guide's LOCAL RELATIVE deviation from its own smoothed value
    (guide / smooth(guide)), instead of adding an unscaled absolute texture
    difference. This is scale-invariant -- it doesn't assume MODIS's local
    pixel-to-pixel swings are the same magnitude as PACE's, which the earlier
    additive version implicitly did (and which produced a result uniformly
    worse than baseline on every test tile -- see notebook history).
    """
    coarse_upsampled = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    guide = np.where(visible_mask, fine_reference, coarse_upsampled)
    guide = np.where(np.isfinite(guide), guide, coarse_upsampled)
    target = np.where(np.isfinite(coarse_upsampled), coarse_upsampled, np.nanmean(coarse_pred))

    guide_smooth = box_filter(guide, radius)
    # Guard against dividing by a near-zero local mean (e.g. bare/non-vegetated
    # ground, where % tree cover is close to 0) -- floor the denominator
    # rather than letting the ratio blow up.
    ratio = guide / np.clip(guide_smooth, 1.0, None)
    result = target * ratio
    return np.clip(result, 0, 100)


def evaluate_holdout_residual(tile: str, radius: int, block_size: int = CHECKERBOARD_BLOCK_SIZE):
    """Same checkerboard-holdout evaluation as evaluate_holdout(), for the
    residual-injection method (single hyperparameter: radius, no eps)."""
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)

    visible_mask = make_checkerboard_mask(TILE_SIZE_PACE, AGGREGATION_FACTOR, block_size)
    held_out_mask = ~visible_mask

    sr_result = apply_super_resolution_residual(coarse_pred, fine_reference, visible_mask, radius)

    valid = held_out_mask & np.isfinite(fine_reference) & np.isfinite(sr_result)
    n_valid = int(np.sum(valid))
    if n_valid == 0:
        return None

    truth = fine_reference[valid]
    pred = sr_result[valid]
    residual = pred - truth

    baseline = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)[valid]
    baseline_residual = baseline - truth

    return {
        "tile": tile,
        "n_valid": n_valid,
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "mae": float(np.mean(np.abs(residual))),
        "bias": float(np.mean(residual)),
        "baseline_rmse": float(np.sqrt(np.mean(baseline_residual ** 2))),
        "baseline_mae": float(np.mean(np.abs(baseline_residual))),
    }

print("Residual/ratio injection functions defined")


In [ ]:
# ## Cell 12: Tune Residual Injection Radius (guide tiles only)

print("=" * 70)
print("RESIDUAL INJECTION -- HYPERPARAMETER SEARCH (guide tiles only)")
print("=" * 70)

residual_search_results = []
for radius in GF_RADIUS_CANDIDATES:
    tile_metrics = [m for m in (evaluate_holdout_residual(tile, radius) for tile in GUIDE_TILES) if m is not None]
    if not tile_metrics:
        continue
    mean_rmse = float(np.mean([m["rmse"] for m in tile_metrics]))
    mean_baseline_rmse = float(np.mean([m["baseline_rmse"] for m in tile_metrics]))
    residual_search_results.append({
        "radius": radius, "mean_rmse": mean_rmse,
        "mean_baseline_rmse": mean_baseline_rmse, "n_tiles": len(tile_metrics),
    })
    print(f"  radius={radius:>3}  RMSE={mean_rmse:6.2f}%  (baseline={mean_baseline_rmse:6.2f}%)  "
          f"n_tiles={len(tile_metrics)}")

best_residual = min(residual_search_results, key=lambda r: r["mean_rmse"])
BEST_RESIDUAL_RADIUS = best_residual["radius"]
print(f"\nBest residual-injection radius: {BEST_RESIDUAL_RADIUS} "
      f"(RMSE={best_residual['mean_rmse']:.2f}% vs. baseline {best_residual['mean_baseline_rmse']:.2f}%)")


In [ ]:
# ## Cell 13: Final Evaluation -- Residual Injection on Test Tiles

print("=" * 70)
print(f"RESIDUAL INJECTION -- FINAL EVALUATION ON TEST TILES (radius={BEST_RESIDUAL_RADIUS})")
print("=" * 70)

residual_test_results = []
for tile in TEST_TILES:
    m = evaluate_holdout_residual(tile, BEST_RESIDUAL_RADIUS)
    if m is None:
        logger.warning(f"  {tile}: no usable held-out data, skipped")
        continue
    m["region"] = tile_regions[tile]
    residual_test_results.append(m)
    print(f"  {tile:<8} ({m['region']:<22}) RMSE={m['rmse']:6.2f}%  "
          f"(baseline {m['baseline_rmse']:6.2f}%)  MAE={m['mae']:6.2f}%  bias={m['bias']:+6.2f}%")

residual_overall_rmse = float(np.mean([m["rmse"] for m in residual_test_results]))
residual_overall_baseline_rmse = float(np.mean([m["baseline_rmse"] for m in residual_test_results]))
print(f"\nMean RMSE:          {residual_overall_rmse:.2f}%")
print(f"Mean baseline RMSE: {residual_overall_baseline_rmse:.2f}%")
print(f"Improvement:        {residual_overall_baseline_rmse - residual_overall_rmse:+.2f} percentage points")


In [ ]:
# ## Cell 14: Three-Way Comparison (Baseline / Guided Filter / Residual Injection)

print("=" * 70)
print("METHOD COMPARISON (test tiles, mean RMSE)")
print("=" * 70)
print(f"  Baseline (no guide):        {overall_baseline_rmse:6.2f}%")
print(f"  Guided filter (r={BEST_RADIUS}, eps={BEST_EPS}):  {overall_rmse:6.2f}%  "
      f"({overall_baseline_rmse - overall_rmse:+.2f}pp vs. baseline)")
print(f"  Residual injection (r={BEST_RESIDUAL_RADIUS}):     {residual_overall_rmse:6.2f}%  "
      f"({residual_overall_baseline_rmse - residual_overall_rmse:+.2f}pp vs. baseline)")


In [ ]:
# ## Cell 15: Product GeoTIFFs -- Guided Filter + Residual Injection (5 standard test tiles)
# Uses ALL available real guide data (no checkerboard holdout) -- this is the
# deliverable product, not an evaluation run.

import rasterio.transform
import rasterio.crs

STANDARD_TEST_TILES = ["h09v05", "h12v04", "h12v09", "h20v06", "h31v11"]
PRODUCT_OUTPUT_DIR = OUTPUT_DIR / "products"
PRODUCT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_sinusoidal_crs():
    return rasterio.crs.CRS.from_proj4(
        "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"
    )


def get_fine_transform(tile: str):
    h = int(tile[1:3])
    v = int(tile[4:6])
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    pixel_size = MODIS_TILE_SIZE_M / TILE_SIZE_MODIS
    return rasterio.transform.from_origin(min_x, max_y, pixel_size, pixel_size)


def write_fine_geotiff(data: np.ndarray, output_path: Path, tile: str, description: str):
    out_data = np.where(np.isfinite(data) & (data >= 0) & (data <= 100),
                         np.round(data).astype(np.uint8), NO_DATA_OUT)
    with rasterio.open(
        output_path, "w", driver="GTiff",
        height=TILE_SIZE_MODIS, width=TILE_SIZE_MODIS, count=1,
        dtype=np.uint8, crs=get_sinusoidal_crs(), transform=get_fine_transform(tile),
        nodata=NO_DATA_OUT, compress="lzw", tiled=True,
    ) as dst:
        dst.write(out_data, 1)
        dst.set_band_description(1, description)


full_visible_mask = np.ones((TILE_SIZE_MODIS, TILE_SIZE_MODIS), dtype=bool)

for tile in STANDARD_TEST_TILES:
    if tile not in AVAILABLE_TILES:
        logger.warning(f"{tile}: not available, skipping product generation")
        continue

    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)
    baseline_full = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)
    gf_result = apply_super_resolution(coarse_pred, fine_reference, full_visible_mask, BEST_RADIUS, BEST_EPS)
    resid_result = apply_super_resolution_residual(coarse_pred, fine_reference, full_visible_mask, BEST_RESIDUAL_RADIUS)

    write_fine_geotiff(baseline_full, PRODUCT_OUTPUT_DIR / f"SuperRes_baseline_{tile}_250m.tif",
                        tile, f"Baseline (nearest upsample) {tile}")
    write_fine_geotiff(gf_result, PRODUCT_OUTPUT_DIR / f"SuperRes_guidedfilter_{tile}_250m.tif",
                        tile, f"Guided filter (r={BEST_RADIUS}, eps={BEST_EPS}) {tile}")
    write_fine_geotiff(resid_result, PRODUCT_OUTPUT_DIR / f"SuperRes_residual_{tile}_250m.tif",
                        tile, f"Residual injection (r={BEST_RESIDUAL_RADIUS}) {tile}")

    print(f"{tile}: wrote baseline, guided-filter, and residual-injection products")

print(f"\nProducts saved to: {PRODUCT_OUTPUT_DIR}")
print("(AlphaEarth-regression product will come from notebook 9b, into the same directory)")


In [ ]:
# ## Cell 16: Sanity Check -- Plain Smoothing, No Guide At All

def apply_smoothed_baseline(coarse_pred: np.ndarray, radius: int) -> np.ndarray:
    """No guide involved at all -- just a local box-smooth of the
    coarse-upsampled prediction. If this performs comparably to the guided
    filter's best setting, that confirms the guided filter's win is coming
    from smoothing scale, not from any real texture borrowed from MODIS."""
    coarse_upsampled = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)
    target = np.where(np.isfinite(coarse_upsampled), coarse_upsampled, np.nanmean(coarse_pred))
    return np.clip(box_filter(target, radius), 0, 100)


def evaluate_holdout_smoothed_baseline(tile: str, radius: int, block_size: int = CHECKERBOARD_BLOCK_SIZE):
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)

    visible_mask = make_checkerboard_mask(TILE_SIZE_PACE, AGGREGATION_FACTOR, block_size)
    held_out_mask = ~visible_mask

    sr_result = apply_smoothed_baseline(coarse_pred, radius)
    baseline_full = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    valid = held_out_mask & np.isfinite(fine_reference) & np.isfinite(sr_result) & np.isfinite(baseline_full)
    n_valid = int(np.sum(valid))
    if n_valid == 0:
        return None

    truth = fine_reference[valid]
    pred = sr_result[valid]
    residual = pred - truth

    baseline = baseline_full[valid]
    baseline_residual = baseline - truth

    return {
        "tile": tile, "n_valid": n_valid,
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "baseline_rmse": float(np.sqrt(np.mean(baseline_residual ** 2))),
    }


print("=" * 70)
print(f"PLAIN SMOOTHING, NO GUIDE (radius={BEST_RADIUS}) -- TEST TILES")
print("=" * 70)

smoothed_test_results = []
for tile in TEST_TILES:
    m = evaluate_holdout_smoothed_baseline(tile, radius=BEST_RADIUS)
    if m is None:
        continue
    smoothed_test_results.append(m)
    print(f"  {tile:<8} RMSE={m['rmse']:6.2f}%  (baseline {m['baseline_rmse']:6.2f}%)")

smoothed_overall_rmse = float(np.mean([m["rmse"] for m in smoothed_test_results]))
smoothed_overall_baseline = float(np.mean([m["baseline_rmse"] for m in smoothed_test_results]))

print(f"\n{'='*70}")
print("COMPARISON")
print(f"{'='*70}")
print(f"  Plain smoothing (no guide):        {smoothed_overall_rmse:.2f}%  "
      f"({smoothed_overall_baseline - smoothed_overall_rmse:+.2f}pp vs. baseline)")
print(f"  Guided filter (radius={BEST_RADIUS}, eps={BEST_EPS}): {overall_rmse:.2f}%  "
      f"({overall_baseline_rmse - overall_rmse:+.2f}pp vs. baseline)")


In [ ]:
# ## Cell 17: Visualize Guided Filter vs. Plain Smoothing (no guide)

def plot_smoothing_comparison(tile: str, radius: int = BEST_RADIUS, eps: float = BEST_EPS,
                               block_size: int = CHECKERBOARD_BLOCK_SIZE,
                               crop_coarse_size: int = 8, crop_row: int = None, crop_col: int = None):
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)
    visible_mask = make_checkerboard_mask(TILE_SIZE_PACE, AGGREGATION_FACTOR, block_size)

    gf_result = apply_super_resolution(coarse_pred, fine_reference, visible_mask, radius, eps)
    smoothed_result = apply_smoothed_baseline(coarse_pred, radius)
    baseline = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    if crop_row is None or crop_col is None:
        found = find_high_contrast_crop(fine_reference, crop_coarse_size)
        if found is not None:
            crop_row, crop_col = found
        else:
            crop_row = (TILE_SIZE_PACE - crop_coarse_size) // 2
            crop_col = (TILE_SIZE_PACE - crop_coarse_size) // 2
    crop_row = max(0, min(crop_row, TILE_SIZE_PACE - crop_coarse_size))
    crop_col = max(0, min(crop_col, TILE_SIZE_PACE - crop_coarse_size))

    fine_row0 = crop_row * AGGREGATION_FACTOR
    fine_row1 = (crop_row + crop_coarse_size) * AGGREGATION_FACTOR
    fine_col0 = crop_col * AGGREGATION_FACTOR
    fine_col1 = (crop_col + crop_coarse_size) * AGGREGATION_FACTOR

    coarse_crop = coarse_pred[crop_row:crop_row + crop_coarse_size, crop_col:crop_col + crop_coarse_size]
    baseline_crop = baseline[fine_row0:fine_row1, fine_col0:fine_col1]
    smoothed_crop = smoothed_result[fine_row0:fine_row1, fine_col0:fine_col1]
    gf_crop = gf_result[fine_row0:fine_row1, fine_col0:fine_col1]
    ref_crop = fine_reference[fine_row0:fine_row1, fine_col0:fine_col1]

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    im0 = axes[0].imshow(coarse_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[0].set_title(f"{tile}: Coarse prediction (2km)\n{crop_coarse_size}x{crop_coarse_size} PACE pixels")
    im1 = axes[1].imshow(baseline_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[1].set_title("Baseline (nearest upsample)")
    im2 = axes[2].imshow(smoothed_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[2].set_title(f"Plain smoothing, no guide (r={radius})")
    im3 = axes[3].imshow(gf_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[3].set_title(f"Guided filter (r={radius}, eps={eps})")
    im4 = axes[4].imshow(ref_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[4].set_title(f"Real MODIS C6 {MODIS_REFERENCE_YEAR} (250m)")
    for ax, im in zip(axes, [im0, im1, im2, im3, im4]):
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="% Tree Cover")
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f"smoothing_comparison_{tile}_crop_r{crop_row}_c{crop_col}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    logger.info(f"Saved: {fig_path}")
    plt.show()


if TEST_TILES:
    plot_smoothing_comparison(TEST_TILES[0])
